In [2]:
import json
from pathlib import Path
from lkb.kb.uriel_plus import URIELPlus
from lkb.eval.splits import Split
from lkb.eval.evaluator import Evaluator

In [3]:
kb = URIELPlus.from_artifacts(
    ".",
    typology_path="../data/derived/URIELPlus_Union.csv",
)

print(f"Languages: {len(kb.languages):,}")
print(f"Features:  {len(kb.features):,}")

obs = kb.observed_mask()
print(f"Observed entries: {obs.sum():,}  ({obs.mean()*100:.1f}% of matrix)")

Languages: 8,171
Features:  800
Observed entries: 888,741  (13.6% of matrix)


In [7]:
with open("../data/splits/splits_v2.json", "r", encoding="utf-8") as f:
    splits_data = json.load(f)
split = Split.from_json(splits_data)
gold_items = Evaluator.gold_items_from_split(kb, split, partition="test")

In [12]:
import numpy as np
from lkb.impute.baselines import RandomImputer, MeanImputer, KNNImputer
from lkb.impute.softimpute import SoftImputeImputer

# Build a masked KB: hide all test + dev pairs so baselines can't see them during fit.
masked_matrix = kb.as_matrix().copy()
_lang_to_row = {lang: i for i, lang in enumerate(kb.languages)}
_feat_to_col = {feat: j for j, feat in enumerate(kb.features)}
for lang, feat in set(split.test) | set(split.dev):
    r, c = _lang_to_row.get(lang), _feat_to_col.get(feat)
    if r is not None and c is not None:
        masked_matrix[r, c] = -1

train_kb = URIELPlus(
    matrix=masked_matrix,
    languages=list(kb.languages),
    features=list(kb.features),
)

# Pairs the baselines need to predict
test_pairs = [(item.language, item.feature) for item in gold_items]

BASELINES = [
    RandomImputer(seed=42),
    MeanImputer(),
    KNNImputer(k=5, metric="cosine"),
    SoftImputeImputer(max_rank=64, input_transform="centered"),
]

baseline_predictions = {}

for baseline in BASELINES:
    print(f"Fitting {baseline.name}…", end=" ", flush=True)
    baseline.fit(train_kb)
    preds = baseline.impute(train_kb, test_pairs)
    baseline_predictions[baseline.name] = preds
    print("done")

Fitting random… done
Fitting mean… done
Fitting knn… done
Fitting softimpute… done


In [13]:
evaluator = Evaluator()
all_reports = {}

for name, preds in baseline_predictions.items():
    report = evaluator.evaluate(gold_items, preds)
    all_reports[name] = report

# Print accuracy by resource group for each baseline
groups = ["lrl", "mrl", "hrl", "overall"]
header = f"{'baseline':<15}" + "".join(f"  {g:>8}" for g in groups)
print(header)
print("-" * len(header))
for name, report in all_reports.items():
    row = f"{name:<15}"
    for g in groups:
        acc = report['metrics'].get(g, {}).get("accuracy", float("nan"))
        row += f"  {acc:>8.3f}"
    print(row)

baseline              lrl       mrl       hrl   overall
-------------------------------------------------------
random              0.715     0.760     0.715     0.730
mean                0.815     0.823     0.780     0.806
knn                 0.873     0.860     0.877     0.870
softimpute          0.895     0.900     0.887     0.894
